# The baseline noise (`bstd`) of each channel, after the 450-sample window

*Draft prepared with Claude Code for A. Villano, 2026-11-02.*

`bstd-distribution.ipynb` and `baseline-window-onset.ipynb` showed that with a baseline window of samples 10 to 500 the `bstd` of the real triggers grew with the pulse height, because the rising edge of the pulse (median onset at sample 483 to 486) was inside the window. The readout of the registry now ends the window at **sample 450** (`registry/readouts.toml`, 2026-11-02), and the eight DCRC1pB-triggered series near -82 V have been reprocessed with it into the same output folder, `~/goldsluice-output/r76-82v-dcrc1pb`.

Here is the `bstd` distribution of every channel again, from the new output: real triggers (blue) against random triggers (grey), the old window (500) against the new one (450) for the real triggers, and the height against `bstd`. The notebook reads `pretrigger_samples` from the manifest of the output and keeps one cache per window, so the old and the new numbers are never mixed. The cache of the 500 window, written by `bstd-distribution.ipynb`, is used for the comparison when it exists. Run with the `darkmatter_cli_env` kernel.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from matplotlib import colors

from goldpan.dataset import Dataset

LETTERS = "ABCDEF"
_here = Path.cwd().resolve()
PROCESSING = next(p / "data_analysis" / "R76" / "processing" for p in [_here, *_here.parents] if (p / "data_analysis" / "R76" / "processing").is_dir())
OUT = Path(Dataset.open(PROCESSING / "r76-82v-dcrc1pb.dataset.toml").spec.output)
manifest = json.loads((OUT / "_goldpan.json").read_text(encoding="utf-8"))
PRE = manifest["streams"]["NAI-S2"]["detector"]["segments"][0]["pretrigger_samples"]
print(f"{OUT}: {len(manifest['streams']['NAI-S2']['files'])} files in the manifest, baseline window = samples 10 to {PRE}")
CACHE = Path.home() / "goldsluice-output" / f"relcal-bstd-height-82v-p{PRE}.npz"
OLD_CACHE = Path.home() / "goldsluice-output" / "relcal-bstd-82v.npz"          # made by bstd-distribution.ipynb from the output with the 500 window

def field(tab, k, name):
    # one quantity of one channel: the channel is a struct column of the Parquet file
    return pc.struct_field(tab[f"Side2p{k}"].combine_chunks(), name).to_numpy(zero_copy_only=False)

if CACHE.exists():
    z = np.load(CACHE)
    series, trigger, bstd, height = z["series"], z["trigger"], z["bstd"], z["height"]
else:
    S_, T_, B_, H_ = [], [], [], []
    for f in sorted((OUT / "NAI-S2").glob("*_F[0-9][0-9][0-9][0-9].parquet")):
        tab = pq.read_table(f, columns=["series", "trigger_type"] + [f"Side2p{k}" for k in LETTERS])
        S_.append(np.asarray(tab["series"].to_pylist())); T_.append(np.asarray(tab["trigger_type"].to_pylist()))
        B_.append(np.column_stack([field(tab, k, "baseline_std") for k in LETTERS]))
        H_.append(np.column_stack([field(tab, k, "current_max") for k in LETTERS]))
    series, trigger, bstd, height = np.concatenate(S_), np.concatenate(T_), np.vstack(B_), np.vstack(H_)
    np.savez(CACHE, series=series, trigger=trigger, bstd=bstd, height=height)
real = trigger == "Unknown"
rand = trigger == "Physics"
print(f"{len(series):,} events in {len(np.unique(series))} series: {real.sum():,} real triggers, {rand.sum():,} random triggers")

## The distribution, channel by channel

Real triggers in blue and random triggers in grey (each normalised), on logarithmic axes. The dotted line is 2 times the median `bstd` of the random triggers. The table gives the percentiles, and the share of the real triggers above that line.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, (ax, k) in enumerate(zip(axes.flat, LETTERS)):
    r, q = bstd[real, i], bstd[rand, i]
    lo, hi = np.percentile(bstd[:, i], [0.01, 99.99])
    bins = np.geomspace(max(lo, 0.3), hi * 1.5, 160)
    ax.hist(q, bins=bins, histtype="stepfilled", color="0.8", density=True, label="random triggers")
    ax.hist(r, bins=bins, histtype="step", color="C0", density=True, lw=1.3, label="real triggers")
    ax.axvline(2 * np.median(q), color="C3", ls=":", lw=1)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_title(f"Side2p{k}", fontsize=11)
    ax.set_xlabel(f"bstd (counts), samples 10 to {PRE}")
    ax.set_ylabel("fraction of events per unit bstd")
axes[0, 0].legend(fontsize=8)
fig.tight_layout()
plt.show()

print(f"{'':>12}" + "".join(f"{'Side2p' + k:>11}" for k in LETTERS))
for name, mask in [("real", real), ("random", rand)]:
    for p in (1, 50, 99, 99.9):
        print(f"{name + ' ' + str(p) + ' %':>12}" + "".join(f"{np.percentile(bstd[mask, i], p):>11.2f}" for i in range(6)))
print(f"{'2 x med.rnd':>12}" + "".join(f"{2 * np.median(bstd[rand, i]):>11.2f}" for i in range(6)))
print(f"{'real above':>12}" + "".join(f"{(bstd[real, i] > 2 * np.median(bstd[rand, i])).mean():>10.1%} " for i in range(6)))

## The old window against the new one

The real triggers with the window to sample 500 (the cache of `bstd-distribution.ipynb`) and to sample 450 (this output), for the same events. If the rising edge was the cause of the shift, the new distribution should sit lower, much closer to the random-trigger one. The last rows of the table give the medians and the share of events above twice the random-trigger median in both.

In [ ]:
if OLD_CACHE.exists():
    old = np.load(OLD_CACHE)
    same = len(old["series"]) == len(series) and (old["series"] == series).all() and (old["trigger"] == trigger).all()
    print("same events in the same order:", same)
else:
    old, same = None, False
    print("no cache of the 500 window: skipped")
if same:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for i, (ax, k) in enumerate(zip(axes.flat, LETTERS)):
        lo, hi = np.percentile(bstd[:, i], [0.01, 99.99])
        bins = np.geomspace(max(lo, 0.3), hi * 1.5, 160)
        ax.hist(bstd[rand, i], bins=bins, histtype="stepfilled", color="0.85", density=True, label="random, 450")
        ax.hist(old["bstd"][real, i], bins=bins, histtype="step", color="C1", density=True, lw=1.2, label="real, window to 500")
        ax.hist(bstd[real, i], bins=bins, histtype="step", color="C0", density=True, lw=1.2, label="real, window to 450")
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_title(f"Side2p{k}", fontsize=11)
        ax.set_xlabel("bstd (counts)"); ax.set_ylabel("fraction of events per unit bstd")
    axes[0, 0].legend(fontsize=8)
    fig.tight_layout()
    plt.show()
    print(f"{'':>22}" + "".join(f"{'Side2p' + k:>11}" for k in LETTERS))
    print(f"{'median, window 500':>22}" + "".join(f"{np.median(old['bstd'][real, i]):>11.2f}" for i in range(6)))
    print(f"{'median, window 450':>22}" + "".join(f"{np.median(bstd[real, i]):>11.2f}" for i in range(6)))
    print(f"{'random median, 450':>22}" + "".join(f"{np.median(bstd[rand, i]):>11.2f}" for i in range(6)))
    thr = [2 * np.median(bstd[rand, i]) for i in range(6)]
    print(f"{'above 2x random, 500':>22}" + "".join(f"{(old['bstd'][real, i] > thr[i]).mean():>10.1%} " for i in range(6)))
    print(f"{'above 2x random, 450':>22}" + "".join(f"{(bstd[real, i] > thr[i]).mean():>10.1%} " for i in range(6)))

## Height against `bstd`

The height of the pulse (`current_max`, counts above the baseline) against `bstd`, real triggers, per channel; the dotted line is 2 times the random-trigger median. A ridge in which `bstd` grows with the height would mean that the edge is still in the window.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, (ax, k) in enumerate(zip(axes.flat, LETTERS)):
    bb = np.geomspace(max(np.percentile(bstd[:, i], 0.01), 0.5), np.percentile(bstd[:, i], 99.99), 100)
    hb = np.geomspace(3, 3000, 100)
    ax.hist2d(bstd[real, i], height[real, i].clip(1), bins=[bb, hb], norm=colors.LogNorm(), cmin=1, cmap="viridis")
    ax.axvline(2 * np.median(bstd[rand, i]), color="C3", ls=":", lw=1)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel(f"bstd (counts), samples 10 to {PRE}"); ax.set_ylabel(f"Side2p{k}: height (counts)")
fig.tight_layout()
plt.show()

## By series

The median `bstd` of the real triggers of each series, per channel, for the new window; a series that stands out is a series whose baseline noise is different.

In [ ]:
sers = sorted(np.unique(series))
print(f"{'series':>14}" + "".join(f"{'Side2p' + k:>11}" for k in LETTERS))
for s in sers:
    m = real & (series == s)
    print(f"{s:>14}" + "".join(f"{np.median(bstd[m, i]):>11.2f}" for i in range(6)))

## Pulse evolution from the threshold to 5 % of the saturated height

Now that the baseline is clean, a fine-grained look at how a pulse grows from just above the noise (the threshold) up to **5 % of the saturated height** of its channel (the saturated heights, in counts: A 1277, B 1095, C 589, D 930, E 680, F 883, from `saturation-over-time.ipynb`: 64, 55, 29, 47, 34 and 44 counts at 5 %). For every channel, twelve steps spaced geometrically in the height, from the **threshold** of the channel up to that 5 %, and at every step one clean pulse. The threshold is where clean pulses begin: the 2nd percentile of the height of the clean pulses of the channel (about 15 to 30 counts; the series trigger on Side2pF with a threshold of 10, and below the threshold there are almost no pulses that peak at the trigger), or 6 times the smoothed noise if that is higher.

* **The trace is filtered with `subtract_line`** (the `goldsluice` filter): the straight line fitted to samples 10 to 450, the baseline window of the reprocessed output, is subtracted from every sample. That takes the baseline off and a slow drift of it with it, so the baseline mean is not subtracted separately, and the same filtered trace is used for the selection and for the plots.
* **Height** is the maximum of the filtered trace smoothed over 16 samples.
* **Time** is measured from the trigger, which is at sample 500 (the window of the baseline ends 50 samples before it).
* **A clean pulse** (the same kind of criteria as in `saturation-over-time.ipynb`, with the thresholds scaled down): the smoothed pulse is above 6 times the smoothed noise of samples 10 to 450 and above 1.3 times the largest value of the smoothed baseline window; its peak is in the first 150 samples after the trigger; in the 800 samples (0.64 ms) after the peak, the decay, the smoothed trace never rises again by more than 5 % of the height plus 6 times the smoothed noise, and after that it never exceeds half the height (or 8 times the noise): no second pulse, no step; and the pulse is below half the saturated height. A stricter rebound over the whole trace left no clean pulses below about 30 counts.
* **The `bstd` cut.** Only pulses whose channel has `bstd` (the processed `baseline_std`, samples 10 to 450) **below the dotted line of the `bstd` plots above**, twice the median `bstd` of the random triggers of that channel (A 4.10, B 6.15, C 3.20, D 6.31, E 3.37, F 3.54 counts), are accepted: a pulse to the right of the line is not used. The cut is on the channel of the pulse shown.
* The pool is 21 dumps (three of each series; `07221113_1159`, which is low in every channel, is left out), read from the raw files in `~/idx`, real triggers only. Of the 10 clean pulses nearest in height to a step one is chosen at random (`SEED`).

The raw trace is grey and the smoothed one blue. The six channels of one step are different events. The cleanest pulses are shown, not typical ones; at the lowest steps a few times the noise is all there is, and some noise may still pass.

In [ ]:
import warnings

import goldpan.channels as pch
from goldpan.adapters import idx
from goldpan.config import DCRC_RUN76
from goldsluice.algorithms import run as run_algorithm

warnings.filterwarnings("once")
RAW = Path.home() / "idx"
SENSORS = {c.name: (c.detector, c.channel) for c in pch.sensor_channels()}
GLITCH, TRIG, SM, N_SAMPLES = 10, 500, 16, 4096
PERIOD_US = 0.8
SATH = np.array([1277, 1095, 589, 930, 680, 883.0])      # the saturated height of each channel (saturation-over-time.ipynb)
H_TOP = 0.05 * SATH                                       # counts: the top of the scan, 5 % of the saturated height
N_STEPS, N_NEAREST, SEED = 12, 10, 1
DUMPS = (10, 20, 30)
DECAY = 800                                               # samples after the peak that are checked for a rebound (0.64 ms)
EXCLUDE = {"07221113_1159"}

BSTD_CUT = np.array([2 * np.median(bstd[rand, i]) for i in range(6)])         # the dotted line of the bstd plots: 2 x the random-trigger median
print("bstd cut (counts): " + ", ".join(f"{k} {c:.2f}" for k, c in zip(LETTERS, BSTD_CUT)))

def smooth(y):
    c = np.cumsum(np.pad(y, ((0, 0), (1, 0))), axis=1)
    return (c[:, SM:] - c[:, :-SM]) / SM                  # moving average, length n - SM + 1

def line_sub(x):
    # the goldsluice filter `subtract_line`: the trace minus the straight line fitted (least squares) to samples 10 to 450, the baseline window.
    # It takes the baseline off and removes a slow drift with it, so the baseline mean is not subtracted separately.
    return np.asarray(run_algorithm("subtract_line", np.asarray(x, dtype=np.float64), glitch_samples=GLITCH, fit_end_samples=PRE)["trace"])

def evolution_metrics(x, sat):
    # x: (events, samples), one channel. Returns the smoothed height, the integral, the smoothed noise and whether the pulse is clean.
    y = line_sub(x)                                       # the trace minus the line fitted to its baseline window
    ys = smooth(y)
    sig_s = y[:, GLITCH:PRE].std(axis=1) / np.sqrt(SM)
    pre = ys[:, GLITCH:PRE - SM].max(axis=1)
    hs = ys.max(axis=1)
    pk = ys.argmax(axis=1)
    t = np.arange(ys.shape[1])[None, :]
    after = (t >= pk[:, None]) & (t < pk[:, None] + DECAY)                 # the decay of the pulse: DECAY samples after its peak
    z = np.where(after, ys, np.inf)
    rebound = np.where(after, ys - np.minimum.accumulate(z, axis=1), 0).max(axis=1)
    late = np.where(t >= pk[:, None] + DECAY, ys, -np.inf).max(axis=1)     # whatever comes after the decay: a second pulse
    peak = pk + SM // 2
    clean = ((hs > 6 * sig_s) & (hs > 1.3 * pre) & (hs < 0.5 * sat) & (peak >= TRIG) & (peak < TRIG + 150)
             & (rebound < 6 * sig_s + 0.05 * hs) & (late < np.maximum(0.5 * hs, 8 * sig_s)))
    return hs, y.sum(axis=1) * PERIOD_US * 1e-6, sig_s, clean

pool = []
for s in sorted(set(np.unique(series)) - EXCLUDE):
    for dn in DUMPS:
        e = idx.read(RAW / f"{s}_F{dn:04d}", detector_name="NAI-S2", channels=SENSORS, config=DCRC_RUN76)
        seg = {name: sg for (name, sg) in e.traces}
        hs, integ, sig, clean = [], [], [], []
        for i, k in enumerate(LETTERS):
            x = np.asarray(e.traces[(f"Side2p{k}", seg[f"Side2p{k}"])], dtype=np.float32)
            a, b, c, d_ = evolution_metrics(x, SATH[i])
            hs.append(a); integ.append(b); sig.append(c); clean.append(d_)
        number = np.asarray(e.quantities["event"])
        tab = pq.read_table(OUT / "NAI-S2" / f"{s}_F{dn:04d}.parquet", columns=["event", "trigger_type"] + [f"Side2p{k}" for k in LETTERS])
        ev_pq = tab["event"].to_numpy()
        order = np.argsort(ev_pq)
        row = order[np.searchsorted(ev_pq[order], number)]                       # the row of the output for each event of the raw dump
        assert (ev_pq[row] == number).all()
        real_ev = np.asarray(tab["trigger_type"].to_pylist())[row] == "Unknown"
        bs = np.column_stack([field(tab, k, "baseline_std")[row] for k in LETTERS])          # the bstd of the processing (samples 10 to 450)
        clean = np.array(clean).T & (bs < BSTD_CUT)                              # the cut: only pulses to the left of the red dotted line
        pool.append(dict(series=s, dump=dn, real=real_ev, hs=np.array(hs).T, integral=np.array(integ).T, sig=np.array(sig).T, clean=clean, bstd=bs))
P = {key: np.concatenate([p[key] for p in pool]) for key in ("real", "hs", "integral", "sig", "clean", "bstd")}
P["dumpref"] = np.concatenate([np.full(len(p["real"]), n) for n, p in enumerate(pool)])
P["row"] = np.concatenate([np.arange(len(p["real"])) for p in pool])
print(f"{len(P['real']):,} events in the pool; clean pulses (real triggers) per channel: " +
      ", ".join(f"{k} {(P['real'] & P['clean'][:, i]).sum():,}" for i, k in enumerate(LETTERS)))

In [ ]:
rng = np.random.default_rng(SEED)
steps = {}
chosen = {}
for i, k in enumerate(LETTERS):
    ok = np.flatnonzero(P["real"] & P["clean"][:, i])
    h_lo = max(6 * np.median(P["sig"][:, i]), np.percentile(P["hs"][ok, i], 2))      # the threshold: where clean pulses begin (2nd percentile)
    steps[i] = np.geomspace(h_lo, H_TOP[i], N_STEPS)
    picks = []
    for target in steps[i]:
        near = ok[np.argsort(np.abs(np.log(P["hs"][ok, i] / target)))[:N_NEAREST]]
        picks.append(int(rng.choice(near)))
    chosen[i] = picks
print("smoothed height (counts) of the pulse chosen at each step (the first step is the threshold of the channel)")
print(f"{'step':>5}" + "".join(f"{'Side2p' + k:>11}" for k in LETTERS))
for s in range(N_STEPS):
    print(f"{s + 1:>5}" + "".join(f"{P['hs'][chosen[i][s], i]:>11.1f}" for i in range(6)))
print(f"{'5 %':>5}" + "".join(f"{H_TOP[i]:>11.1f}" for i in range(6)) + "   (top of the scan: 5 % of the saturated height)")
print(f"{'noise':>5}" + "".join(f"{np.median(P['sig'][:, i]) * 4:>11.1f}" for i in range(6)) + "   (standard deviation of the baseline)")

# the traces of the chosen pulses: read their dumps once more
trace = {}
by_dump = {}
for i in range(6):
    for j in chosen[i]:
        by_dump.setdefault(int(P["dumpref"][j]), set()).add(j)
for n, rows in sorted(by_dump.items()):
    p = pool[n]
    e = idx.read(RAW / f"{p['series']}_F{p['dump']:04d}", detector_name="NAI-S2", channels=SENSORS, config=DCRC_RUN76)
    seg = {name: sg for (name, sg) in e.traces}
    for j in rows:
        trace[j] = np.array([np.asarray(e.traces[(f"Side2p{k}", seg[f"Side2p{k}"])][int(P["row"][j])], dtype=float) for k in LETTERS])
t_us = (np.arange(N_SAMPLES) - TRIG) * PERIOD_US
t_s = t_us[SM // 2: SM // 2 + N_SAMPLES - SM + 1]

def flat(j, i):
    return line_sub(trace[j][i][None, :])[0]              # the same filter as in the selection

def smooth1(y):
    return smooth(y[None, :])[0]

In [ ]:
fig, axes = plt.subplots(6, N_STEPS, figsize=(2.05 * N_STEPS, 11), sharex=True)
for i, k in enumerate(LETTERS):
    for s in range(N_STEPS):
        ax = axes[i, s]
        j = chosen[i][s]
        f = flat(j, i)
        ax.axvline(0, color="0.6", ls=":", lw=0.7)
        ax.plot(t_us, f, lw=0.4, color="0.6")
        ax.plot(t_s, smooth1(f), lw=0.9, color="C0")
        top = 1.5 * P["hs"][j, i]
        ax.set_ylim(-0.5 * top, top)
        ax.set_xlim(-100, 1500)
        ax.set_title(f"{P['hs'][j, i]:.0f} counts\n{100 * P['hs'][j, i] / SATH[i]:.1f} % of sat.", fontsize=7)
        if s == 0:
            ax.set_ylabel(f"Side2p{k}\ncounts")
for s in range(N_STEPS):
    axes[0, s].text(0.5, 1.45, f"step {s + 1}", transform=axes[0, s].transAxes, ha="center", fontsize=8, weight="bold")
    axes[-1, s].set_xlabel("us from the trigger", fontsize=8)
fig.tight_layout()
plt.show()

### The same as a movie

One frame per step, the six channels side by side with a vertical scale set by each pulse. Saved as `figures/pulse-evolution-450.gif`. The text in each panel gives the smoothed height, its share of the saturated height, and the integral (counts x s, of the filtered trace over the whole length).

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image

FIG_DIR = Path.cwd() / "figures"
FIG_DIR.mkdir(exist_ok=True)
fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), sharex=True)
raw_l, sm_l, texts = [], [], []
for ax, i, k in zip(axes.flat, range(6), LETTERS):
    ax.axvline(0, color="0.6", ls=":", lw=0.7)
    ax.axhline(0, color="0.8", lw=0.5)
    ax.set_title(f"Side2p{k}", fontsize=11)
    ax.set_xlim(-100, 1500)
    (a,) = ax.plot([], [], lw=0.4, color="0.6")
    (b,) = ax.plot([], [], lw=1.2, color="C0")
    raw_l.append(a); sm_l.append(b)
    texts.append(ax.text(0.97, 0.93, "", transform=ax.transAxes, ha="right", va="top", fontsize=9))
for ax in axes[-1]:
    ax.set_xlabel("time from the trigger (us)")
for ax in axes[:, 0]:
    ax.set_ylabel("counts above baseline")
head = fig.suptitle("", fontsize=12)

def frame(s):
    head.set_text(f"step {s + 1} of {N_STEPS}: from the threshold to 5 % of the saturated height")
    for i, ax in zip(range(6), axes.flat):
        j = chosen[i][s]
        f = flat(j, i)
        raw_l[i].set_data(t_us, f)
        sm_l[i].set_data(t_s, smooth1(f))
        top = 1.5 * P["hs"][j, i]
        ax.set_ylim(-0.5 * top, top)
        texts[i].set_text(f"{P['hs'][j, i]:.0f} counts ({100 * P['hs'][j, i] / SATH[i]:.1f} %)\nintegral {P['integral'][j, i]:.3f}")
    return raw_l + sm_l + texts

fig.tight_layout(rect=(0, 0, 1, 0.95))
anim = FuncAnimation(fig, frame, frames=N_STEPS, interval=800, blit=False)
GIF = FIG_DIR / "pulse-evolution-450.gif"
anim.save(GIF, writer=PillowWriter(fps=1.25), dpi=70)
plt.close(fig)
Image(filename=str(GIF))

**What this shows, and what is not done.** The `bstd` of every channel with the baseline window ending at sample 450, against the old window and against the random triggers. Not done: a quality cut on `bstd`; the choice (a multiple of the random-trigger median, a percentile, or a limit per series) follows from these plots. The processing has no status file yet (that is `goldsluice run --replace` and the status file of pull request 201), so that the output was completely rewritten is known only from its 436 files in the manifest.